<div style="background-color:#DFC7F2; padding:15px; margin:15px;">


**Центр непрерывного образования НИУ ВШЭ**

## Программа «Python для автоматизации и анализа данных»

### Занятие 10. Часть 1. Классы на практике: метод k-ближайших соседей

*Автор: Маргарита Бурова, НИУ ВШЭ*

</div>

# Метод k-ближайших соседей (kNN)

Давайте перейдём к более прикладным задачам: сначала познакомимся с методом k-ближайших соседей, а потом реализуем его в виде класса.

`kNN` расшифровывается как `k Nearest Neighbors`, или «k ближайших соседей», — это один из самых простых алгоритмов классификации (иногда его применяют и в задачах регрессии). Задача классификации в машинном обучении — это задача отнесения объекта к одному из заранее определённых классов на основе его формализованных признаков. Каждый объект в этой задаче представляется в виде вектора в N-мерном пространстве, где каждое измерение — это один из признаков объекта. Например, если мы классифицируем телевизоры, измерениями могут быть диагональ в дюймах, максимальное разрешение, наличие HDMI-интерфейса, стоимость и т. д.

## Теория

Идея алгоритма kNN довольно проста. Он считает расстояние от новой точки до всех известных точек обучающей выборки. Расстояние можно считать разными способами. Затем алгоритм выбирает k ближайших точек, где k — заранее заданное целое число. И в конце новой точке присваивается тот класс, к которому относится большинство из этих k соседей.

## Датасет

В качестве датасета мы будем использовать знаменитый набор данных «Ирисы» (iris). Он состоит из четырёх признаков: «длина чашелистика», «ширина чашелистика», «длина лепестка» и «ширина лепестка». В нём представлены три вида ириса: `Iris-setosa` (ирис щетинистый), `Iris-versicolor` (ирис разноцветный) и `Iris-virginica` (ирис виргинский).

У каждого растения есть четыре признака: `sepal-length` (длина чашелистика), `sepal-width` (ширина чашелистика), `petal-length` (длина лепестка) и `petal-width` (ширина лепестка).

Импортируем датасет и загрузим его в `pandas`:

In [ ]:
import numpy as np
import pandas as pd

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data"

# Назовём колонки датасета
names = ['sepal-length', 'sepal-width', 'petal-length', 'petal-width', 'Class']

# Загрузим датасет
dataset = pd.read_csv(url, names=names)

Давайте посмотрим, как он выглядит и какие в нём есть значения:

In [ ]:
dataset.head()

Дальше разобьём датасет на признаки (`фичи`) и `метки`. У нас 4 столбца с признаками, но предсказывать мы будем только по первым двум.

In [ ]:
X = dataset.iloc[:, :-3].values
y = dataset.iloc[:, 4].values

In [ ]:
X

In [ ]:
y

Массив `X` содержит значения первых двух колонок, а `y` — метки классов. Предсказывать мы будем по `sepal-length` и `sepal-width`.

Давайте визуализируем данные с помощью `matplotlib`:

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

plt.figure()
plt.scatter(X[:, 0], X[:, 1])
plt.title("Длина и ширина чашелистиков ириса")
plt.show()

### Разделение на обучающую и тестовую выборки

Чтобы избежать переобучения, мы разделим датасет на обучающую и тестовую выборки. Это позволит понять, насколько хорошо работает наш алгоритм: он проверяется на «незнакомых» ему данных, а значит, его можно будет применять и к реальным задачам.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Мы использовали функцию `train_test_split` из модуля `sklearn`. Код выше разделил датасет на 80% обучающих данных и 20% тестовых. Это значит, что из 150 записей в обучающую выборку попадёт 120, а в тестовую — 30.

In [ ]:
X_train

In [ ]:
print(len(X_train), len(y_train))

In [ ]:
print(len(X_test), len(y_test))

## Простая реализация kNN

Теперь, когда у нас есть обучающая выборка, можно реализовать сам метод классификации:

In [ ]:
def dist(a, b):
    return ((a[0] - b[0])**2 + (a[1] - b[1])**2)**0.5

Для определения расстояния от искомой точки до соседей мы будем использовать «евклидово расстояние». Но это не единственный вариант: можно также использовать манхэттенское расстояние, косинусную меру, коэффициент корреляции Пирсона и т. д.

In [ ]:
def predict(X_train, y_train, x_test, k):
    # создадим список для расстояний и словарь для меток k ближайших соседей
    distances = []
    targets = {}

    for i in range(len(X_train)):
        # пройдём по всем точкам и посчитаем расстояние до них от тестовой точки
        distances.append([dist(x_test, X_train[i]), i])

    # отсортируем расстояния
    distances = sorted(distances)

    # создадим словарь с метками ближайших соседей
    for i in range(k):
        index = distances[i][1]
        if targets.get(y_train[index]) != None:
            targets[y_train[index]] += 1
        else:
            targets[y_train[index]] = 1

    # вернём самую часто встречающуюся метку
    return max(targets, key=targets.get)

Сначала мы создаём список `distances` и словарь `targets`. В `distances` будем хранить расстояния от искомой точки до всех остальных вместе с индексом каждой точки. А в `targets` — метки `k` ближайших соседей нашей искомой точки.

Найдя все расстояния, мы сортируем список по возрастанию. Потом берём первые `k` точек из начала списка `distances` и считаем, сколько раз в них встречается каждая метка.

И возвращаем метку, которая встречается чаще всего.

In [ ]:
# Создадим список для будущих предсказаний
predictions = []
# количество соседей
k = 7
# идём по всем тестовым данным
for i in range(len(X_test)):
    predictions.append(predict(X_train, y_train, X_test[i], k))

Давайте посмотрим, какие предсказания мы получили:

In [ ]:
print(predictions)

А вот что должно было получиться:

In [ ]:
print(y_test)

Визуализируем исходные метки. Для этого переведём текстовые значения меток в числа:

In [ ]:
colors = []
for c in y:
    if c == "Iris-setosa":
        colors.append(0)
    elif c == "Iris-versicolor":
        colors.append(1)
    else:
        colors.append(2)

Весь датасет:

In [ ]:
plt.scatter(X[:, 0], X[:, 1], c=colors)
plt.title("Датасет Iris")
plt.show()

Раскрасим предсказанные и настоящие метки тестовой выборки:

In [ ]:
colors = []
for c in predictions:
    if c == "Iris-setosa":
        colors.append(0)
    elif c == "Iris-versicolor":
        colors.append(1)
    else:
        colors.append(2)

colors_actual = []
for c in y_test:
    if c == "Iris-setosa":
        colors_actual.append(0)
    elif c == "Iris-versicolor":
        colors_actual.append(1)
    else:
        colors_actual.append(2)

In [ ]:
plt.scatter(X_test[:, 0], X_test[:, 1], c=colors)
plt.title("Iris Predicted")
plt.show()

In [ ]:
plt.scatter(X_test[:, 0], X_test[:, 1], c=colors_actual)
plt.title("Iris Actual")
plt.show()

А теперь посчитаем, насколько качественно работает наш алгоритм:

In [ ]:
correct = 0
for i in range(len(y_test)):
    if y_test[i] == predictions[i]:
        correct += 1

print(correct, '/', len(y_test))
correct / len(y_test)

Из 30 предсказаний 22 оказались верными, то есть точность классификации — 73%.

## Реализация kNN в виде класса

У нас уже есть все нужные функции, поэтому реализуем kNN в виде класса. Он будет состоять из конструктора, где мы создадим атрибуты, метода `_dist`, который возвращает евклидово расстояние, метода `fit`, который подготовит наш классификатор, и метода `predict`, который будет возвращать предсказанные значения. Также создадим методы `score` и `plot`.

In [ ]:
class KNN:
    """
    Классификатор k-NN

    Возвращает: предсказания k-NN
    """

    def __init__(self):
        self.X_train = None
        self.y_train = None
        self.k = None
        self.X_test = None
        self.predictions = []

    @staticmethod
    def _dist(a, b):
        """
        Евклидово расстояние
        Принимает на вход два вектора

        Возвращает: число с плавающей точкой
        """
        return ((a[0] - b[0])**2 + (a[1] - b[1])**2)**0.5

    @staticmethod
    def score(y_test, predictions):
        """
        Принимает на вход два массива с данными: тестовый и предсказанный

        Возвращает: число с плавающей точкой
        """
        correct = 0
        for i in range(len(y_test)):
            if y_test[i] == predictions[i]:
                correct += 1
        return correct / len(y_test)

    def plot(self, y_test):
        colors = []
        for c in y_test:
            if c == "Iris-setosa":
                colors.append(0)
            elif c == "Iris-versicolor":
                colors.append(1)
            else:
                colors.append(2)

        plt.scatter(self.X_test[:, 0], self.X_test[:, 1], c=colors)
        plt.title("Реальные данные")
        plt.show()

        colors = []
        for c in self.predictions:
            if c == "Iris-setosa":
                colors.append(0)
            elif c == "Iris-versicolor":
                colors.append(1)
            else:
                colors.append(2)

        plt.scatter(self.X_test[:, 0], self.X_test[:, 1], c=colors)
        plt.title("Предсказанные данные")
        plt.show()

    def fit(self, X_train, y_train, k):
        """
        Принимает на вход два массива с данными: тренировочные X и тренировочные метки,
        и k - количество соседей
        """
        self.X_train = X_train
        self.y_train = y_train
        self.k = k

    def predict(self, X_test):
        """
        Принимает на вход двумерный массив искомых точек

        Возвращает: список предсказаний
        """
        self.X_test = X_test
        self.predictions = []

        for i in range(len(X_test)):
            distances = []
            targets = {}

            for j in range(len(self.X_train)):
                # пройдём по всем точкам и посчитаем расстояние до них от тестовой точки
                distances.append([self._dist(X_test[i], self.X_train[j]), j])

            # отсортируем расстояния
            distances = sorted(distances)

            # создадим словарь с метками ближайших соседей
            for j in range(self.k):
                index = distances[j][1]
                if targets.get(self.y_train[index]) != None:
                    targets[self.y_train[index]] += 1
                else:
                    targets[self.y_train[index]] = 1

            # добавим самую часто встречающуюся метку
            self.predictions.append(max(targets, key=targets.get))

        return self.predictions

А теперь попробуем его в деле на том же датасете:

In [ ]:
knn = KNN()
knn.fit(X_train, y_train, 7)

pred = knn.predict(X_test)
print(pred)
print(knn.score(y_test, pred))

В нашем классе мы использовали статические методы `_dist` и `score`. Давайте построим график с помощью метода экземпляра `plot`:

In [ ]:
knn.plot(y_test)

## Класс kNN с другой функцией расстояния

Давайте создадим второй класс kNN, который будет использовать другую функцию расстояния. Писать второй класс с нуля не будем, а воспользуемся свойством классов — наследованием. В новом классе поменяем только функцию расстояния, а всё остальное оставим как было. Для вычисления косинусной меры используем функцию `cosine` из библиотеки `scipy`.

In [ ]:
from scipy.spatial import distance

class cosKNN(KNN):

    @staticmethod
    def _dist(a, b):
        """
        Косинусная мера

        Возвращает: число с плавающей точкой
        """
        return distance.cosine(a, b)

In [ ]:
cos_kNN = cosKNN()
cos_kNN.fit(X_train, y_train, 7)
pred = cos_kNN.predict(X_test)
print(cosKNN.score(y_test, pred))

In [ ]:
cos_kNN.plot(y_test)

## Класс kNN с поддержкой нескольких функций расстояния

Выше мы создали два класса: наследник отличался от родителя только функцией расстояния (метрикой). Но всё это можно реализовать и в рамках одного класса. Давайте применим наши знания о классах и создадим такой класс, заодно изменив методы `score` и `_dist`.

In [ ]:
class ultimateKNN:
    """
    Классификатор k-NN

    Возвращает: предсказания k-NN
    """

    def __init__(self):
        self.X_train = None
        self.y_train = None
        self.k = None
        self.metric = None
        self.X_test = None
        self.predictions = []

    def __dist(self, a, b):
        """
        Евклидово расстояние или косинусная мера
        Принимает на вход два вектора

        Возвращает: число с плавающей точкой
        """
        if self.metric == "eucl":
            return ((a[0] - b[0])**2 + (a[1] - b[1])**2)**0.5
        elif self.metric == "cos":
            return distance.cosine(a, b)

    def score(self, y_test):
        """
        Принимает на вход массив с правильными метками
        и сравнивает его с сохранёнными предсказаниями

        Возвращает: число с плавающей точкой
        """
        k = 0
        for i in range(len(y_test)):
            if y_test[i] == self.predictions[i]:
                k += 1
        return k / len(y_test)

    def plot(self, y_test):
        colors = []
        for c in y_test:
            if c == "Iris-setosa":
                colors.append(0)
            elif c == "Iris-versicolor":
                colors.append(1)
            else:
                colors.append(2)
        Xx = []
        Yy = []
        for i in self.X_test:
            Xx.append(i[0])
            Yy.append(i[1])

        plt.scatter(Xx, Yy, c=colors)
        plt.title("Iris Real Data")
        plt.show()

        colors = []
        for c in self.predictions:
            if c == "Iris-setosa":
                colors.append(0)
            elif c == "Iris-versicolor":
                colors.append(1)
            else:
                colors.append(2)

        plt.scatter(Xx, Yy, c=colors)
        plt.title("Iris Predicted")
        plt.show()

    def fit(self, X_train, y_train, k, metric="eucl"):
        """
        Принимает на вход два массива с данными: тренировочные X и тренировочные метки,
        k - количество соседей и metric - метрику ("eucl" или "cos")
        """
        self.X_train = X_train
        self.y_train = y_train
        self.k = k
        self.metric = metric

    def predict(self, X_test):
        """
        Принимает на вход двумерный массив искомых точек

        Возвращает: список предсказаний
        """
        self.X_test = X_test
        self.predictions = []

        for i in range(len(X_test)):
            distances = []
            targets = {}

            for j in range(len(self.X_train)):
                # пройдём по всем точкам и посчитаем расстояние до них от тестовой точки
                distances.append([self.__dist(X_test[i], self.X_train[j]), j])

            # отсортируем расстояния
            distances = sorted(distances)

            # создадим словарь с метками ближайших соседей
            for j in range(self.k):
                index = distances[j][1]
                if targets.get(self.y_train[index]) != None:
                    targets[self.y_train[index]] += 1
                else:
                    targets[self.y_train[index]] = 1

            # добавим самую часто встречающуюся метку
            self.predictions.append(max(targets, key=targets.get))

        return self.predictions

In [ ]:
ult_kNN = ultimateKNN()
ult_kNN.fit(X_train, y_train, 7)
print(ult_kNN.predict(X_test))
print(ult_kNN.score(y_test))

In [ ]:
ult_kNN.plot(y_test)

Класс в зависимости от указанной метрики выбирает способ подсчёта расстояния до точки. Если мы ничего не задаём или задаём `eucl`, расстояние считается как евклидово, а если `cos` — через косинусную меру.

In [ ]:
ult_kNN = ultimateKNN()
ult_kNN.fit(X_train, y_train, 7, 'cos')
print(ult_kNN.predict(X_test))
print(ult_kNN.score(y_test))

In [ ]:
ult_kNN.plot(y_test)

Давайте теперь переберём параметр: построим несколько kNN-классификаторов с разными `k` и посмотрим, какой из них лучше:

In [ ]:
for i in range(1, 10):
    knn = ultimateKNN()
    knn.fit(X_train, y_train, i)
    knn.predict(X_test)
    print("k = " + str(i), ", Score: " + str(knn.score(y_test)))

In [ ]:
for i in range(1, 10):
    knn = ultimateKNN()
    knn.fit(X_train, y_train, i, "cos")
    knn.predict(X_test)
    print("k = " + str(i), ", Score: " + str(knn.score(y_test)))

Как мы видим, на нашем датасете евклидово расстояние работает лучше косинусной меры.

# Заключение

Мы познакомились с классами и написали классы, реализующие метод kNN. kNN — один из простейших алгоритмов классификации, но у него есть свои недостатки. Помимо точности классификации, проблема этого классификатора — скорость работы: если в обучающей выборке `N` объектов, в тестовой выборке `M` объектов, а размерность пространства — `K`, то количество операций для классификации тестовой выборки можно оценить как `O(K*M*N)`.